# Stage 5 — Model building (LightGBM)

Starting simple, tuning hyperparameters one at a time with explanation.

## Step 1: Train/validation data (Fold 0 cutoff)

Reusing Fold 0's cutoff from Stage 4. LightGBM handles our lag/rolling/
price NaNs natively (learns a default split direction for missing
values), so no need to drop or impute those rows. categorical_feature
tells LightGBM which integer columns are unordered labels, not numbers.


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from pathlib import Path

BASE_DIR = Path("/Users/asadkhan.pathan/Documents/New project/m5 forecasting ")
df = pd.read_parquet(BASE_DIR / "data" / "processed" / "features.parquet")
df = df.sort_values(["id", "date"]).reset_index(drop=True)

HORIZON = 28
cutoff = df["date"].max() - pd.Timedelta(days=HORIZON)

train = df[df["date"] <= cutoff]
valid = df[df["date"] > cutoff]

cat_features = [
    "item_id_enc", "dept_id_enc", "cat_id_enc", "store_id_enc", "state_id_enc",
    "event_name_1_enc", "event_type_1_enc", "event_name_2_enc", "event_type_2_enc",
]
num_features = [
    "wday", "month", "year", "is_weekend", "snap_active", "has_event",
    "lag_28", "lag_35", "lag_42", "lag_49", "lag_56",
    "rolling_mean_7_lag28", "rolling_mean_28_lag28",
    "sell_price", "price_change_vs_last_week", "price_rel_to_own_avg",
]
features = cat_features + num_features
target = "sales"

train_set = lgb.Dataset(train[features], label=train[target], categorical_feature=cat_features, free_raw_data=False)
valid_set = lgb.Dataset(valid[features], label=valid[target], categorical_feature=cat_features, reference=train_set, free_raw_data=False)

print("train rows:", len(train), " valid rows:", len(valid))
print("num features:", len(features), " categorical:", len(cat_features))


train rows: 58327370  valid rows: 853720
num features: 25  categorical: 9


In [3]:
params = {
    "objective": "regression",
    "metric": "rmse",
    "num_leaves": 31,
    "learning_rate": 0.1,
    "min_data_in_leaf": 50,
    "verbose": -1,
    
}

model = lgb.train(
    params,
    train_set,
    num_boost_round=2000,
    valid_sets=[valid_set],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=50)],
)

print("best iteration:", model.best_iteration)
print("best valid RMSE:", model.best_score["valid_0"]["rmse"])


[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
Training until validation scores don't improve for 30 rounds
[50]	valid_0's rmse: 2.194
[100]	valid_0's rmse: 2.17695
[150]	valid_0's rmse: 2.16938
[200]	valid_0's rmse: 2.16442
[250]	valid_0's rmse: 2.15663
[300]	valid_0's rmse: 2.15206
[350]	valid_0's rmse: 2.149
[400]	valid_0's rmse: 2.14679
Early stopping, best iteration is:
[399]	valid_0's rmse: 2.14677
best iteration: 399
best valid RMSE: 2.1467736252613374


## Step 4: Tweedie objective (built for zero-inflated demand)

Stage 1 found 80-95% of days are zero-sales. Squared-error regression
doesn't know that structure. Tweedie models a zero-inflated, right-skewed
target natively — comparing it against Step 3's regression baseline
(RMSE 2.14677) tells us how much this mismatch was costing us.


In [4]:
params_tweedie = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.2,
    "metric": "rmse",
    "num_leaves": 31,
    "learning_rate": 0.1,
    "min_data_in_leaf": 50,
    "verbose": -1,
}

model_tweedie = lgb.train(
    params_tweedie,
    train_set,
    num_boost_round=2000,
    valid_sets=[valid_set],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=50)],
)

print("best iteration:", model_tweedie.best_iteration)
print("best valid RMSE:", model_tweedie.best_score["valid_0"]["rmse"])


Training until validation scores don't improve for 30 rounds
[50]	valid_0's rmse: 2.18885
[100]	valid_0's rmse: 2.1754
Early stopping, best iteration is:
[110]	valid_0's rmse: 2.17409
best iteration: 110
best valid RMSE: 2.1740896639690273


## Step 5: Save validation predictions (for Stage 6 evaluation)

Saving id, date, actual sales, and this model's prediction on the
validation set to a file — so the evaluation notebook (Stage 6) can
compute per-series RMSSE without needing to retrain/reload the model.


In [5]:
valid_preds = model.predict(valid[features], num_iteration=model.best_iteration)

valid_out = valid[["id", "date", "sales"]].copy()
valid_out["pred"] = valid_preds

valid_out.to_parquet(BASE_DIR / "data" / "processed" / "valid_predictions.parquet")
print(valid_out.shape)
print(valid_out.head())


(853720, 4)
                                 id       date  sales      pred
1913  HOBBIES_1_001_CA_1_evaluation 2016-04-25      0  0.825455
1914  HOBBIES_1_001_CA_1_evaluation 2016-04-26      0  0.734764
1915  HOBBIES_1_001_CA_1_evaluation 2016-04-27      0  0.655601
1916  HOBBIES_1_001_CA_1_evaluation 2016-04-28      2  0.613430
1917  HOBBIES_1_001_CA_1_evaluation 2016-04-29      0  0.647892


## Step 2: Per-series RMSSE

Combine Step 5's validation predictions (from 05_lightgbm.ipynb) with
Step 1's per-series scale. RMSSE = sqrt(mean squared error / scale) —
each series' error judged against its own naive-forecast difficulty.


NameError: name 'scale' is not defined